# 第26章　実装ガイド（検出）：YOLOで病変を見つける

**『医療診断支援AI開発　第1巻　入門編 ― 動かして、自分で作る（第1巻　入門編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-vol1

## 26.1　データを準備する：YOLO形式

```text
fracture/
├── images/{train,val}/*.png
├── labels/{train,val}/*.txt    # 例: "0 0.52 0.48 0.11 0.09"
└── fracture.yaml
```

```yaml
path: ./fracture
train: images/train
val:   images/val
names:
  0: fracture
```

## 学習・検証・テストの3分割と、fracture.yamlの設定例

```text
fracture/
├── images/{train,val,test}/*.png
├── labels/{train,val,test}/*.txt
└── fracture.yaml
```

```yaml
path: /data/fracture      # 絶対パスが安全（相対パスは実行ディレクトリ依存でファイルが見つからなくなりやすい）
train: images/train
val:   images/val
test:  images/test        # model.val(split="test") で最終評価にだけ使う
names:
  0: fracture             # クラス番号は0始まり。ラベルtxtの先頭列と一致させる
```

In [ ]:
import shutil
from pathlib import Path
for _, r in df.iterrows():                       # df に split 列（train/val/test）がある前提
    img_dir = Path("fracture/images")/r["split"]; img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir = Path("fracture/labels")/r["split"]; lbl_dir.mkdir(parents=True, exist_ok=True)
    # ファイル名をそのまま使わない。多施設・複数データセットでは同名の画像が普通にあり、
    # 警告なしに上書きされる（画像とラベルを別々にコピーするので、取り違えにもなる）。
    stem = str(r["case_id"])                     # manifestのcase IDで一意な名前を作る
    dst_i = img_dir / (stem + Path(r["image_path"]).suffix)
    dst_l = lbl_dir / (stem + Path(r["label_path"]).suffix)
    assert not dst_i.exists() and not dst_l.exists(), f"出力名が衝突: {stem}"
    shutil.copy(r["image_path"], dst_i)
    shutil.copy(r["label_path"], dst_l)          # 画像とラベルを同じ stem で対応させる

## YOLOのログと results.csv を読む

In [ ]:
import pandas as pd
h = pd.read_csv("runs/detect/train/results.csv")
h.columns = h.columns.str.strip()               # 列名の前後に空白が含まれる場合に備えて除去（列名の不一致によるKeyErrorを防ぐ）
print(h[["epoch", "metrics/recall(B)", "metrics/mAP50(B)"]].tail())

## 26.2　学習する

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")                 # 事前学習済みの軽量モデル
model.train(
    data="fracture/fracture.yaml",
    epochs=100, imgsz=640, batch=16,
    patience=20,                            # 20エポック改善しなければ早期終了
)

## 26.3　評価する

In [ ]:
metrics = model.val()                       # mAPなどを計算
print(metrics.box.map)                       # mAP@[.5:.95]
print(metrics.box.map50)                     # mAP@0.5

## 26.4　推論する：閾値は目的に合わせて

In [ ]:
results = model("new_xray.png", conf=0.10)   # 既定は0.25。検証で選んだ値を明示する（0.10は例）
results[0].show()                             # 枠つきで表示
for box in results[0].boxes:
    print(box.cls, box.conf, box.xyxy)        # クラス・確信度・座標

## 推論の後処理：NMSを改良した方法（Soft-NMSと重み付き枠融合）

In [ ]:
def wbf_fuse(boxes, scores):                 # 同一病変に対応づいた枠群を融合
    w = scores / scores.sum()                # 確信度を重みに正規化
    fused_box = (boxes * w[:, None]).sum(0)  # 座標を確信度で加重平均
    fused_score = scores.mean()              # 融合枠の確信度（平均や重み付き平均）
    return fused_box, fused_score

## 26.5　現場へ届ける

In [ ]:
model.export(format="onnx")                   # ONNXへ書き出し（相互運用・高速推論）